# ASCENT-G Phase 1 Full Run: AMC × Qwen2.5-1.5B-Instruct

**Phase:** 1 — Full 1000-step registered run  
**Model:** `Qwen2.5-1.5B-Instruct` (registered primary)  
**Task:** AMC  
**Method:** GRPO via TRL  

**Deviation from v1.3:** `max_completion_length=96` (registered default: 256).  
Rationale: 64-token run (v5) showed near-zero reward — reasoning truncated before final answer. 96 tokens allows minimal chain-of-thought while keeping step time manageable (~30s/step, ~8–9h total).  
This deviation is logged in the run report and must be labeled as such in all downstream analysis.

**Acceptance criteria (from execution-plan.md):**
1. Model loads successfully
2. GRPO training runs to planned stopping point without environment failure
3. Adapter weights saved and reloadable
4. Non-degenerate update vector extractable
5. Update vector consumable by analysis code
6. At least one SVD diagnostic runs successfully

Do not interpret reward/accuracy numbers as registered H1a/H1b/H2 evidence without the full 10-task matrix.

## 1. Environment smoke test

In [ ]:
import subprocess, sys, torch

# Capture pre-installed torch version to use as hard pin.
torch_ver = torch.__version__          # e.g. "2.10.0+cu128"
torch_base = torch_ver.split("+")[0]  # strip CUDA suffix for pip constraint

# 1. Upgrade torchao without touching torch (needed by peft >= 0.18).
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torchao>=0.16.0", "--no-deps"], check=True)

# 2. Install ML packages; constrain torch family to the pre-installed versions.
constraints = (
    f"torch=={torch_base}\n"
    "torchvision\n"
    "torchaudio\n"
    "torchao\n"
).encode()
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "trl==0.17.0", "transformers==4.51.3",
                "peft==0.14.0", "accelerate", "datasets",
                "--constraint", "/dev/stdin"],
               input=constraints, check=True)

# Re-import after installs to pick up new versions.
import importlib
for mod in ["trl", "transformers", "peft", "datasets"]:
    importlib.invalidate_caches()

import trl, transformers, peft, datasets
print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"trl          : {trl.__version__}")
print(f"peft         : {peft.__version__}")
print(f"datasets     : {datasets.__version__}")
print(f"CUDA avail   : {torch.cuda.is_available()}")

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU detected — abort"

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
bf16_supported = torch.cuda.is_bf16_supported()

print(f"GPU model  : {gpu_name}")
print(f"VRAM       : {vram_gb:.1f} GB")
print(f"bf16       : {'supported' if bf16_supported else 'NOT supported — will use float16'}")
print(f"torch      : {torch.__version__}")

# Fail fast if assigned GPU is incompatible with torch 2.10+cu128 (sm_60 not supported).
# P100 (sm_60) will crash at LoRA dtype cast regardless of workarounds.
# Request T4 via Kaggle session settings before running.
REQUIRED_GPU = "T4"
if REQUIRED_GPU not in gpu_name:
    raise RuntimeError(
        f"Wrong GPU: got '{gpu_name}', need {REQUIRED_GPU}. "
        f"Go to Session options → Accelerator → GPU T4 x2 and re-run."
    )

HW_META = {
    "gpu_model": gpu_name,
    "vram_gb": round(vram_gb, 1),
    "bf16_supported": bf16_supported,
    "torch_version": torch.__version__,
}
print("\nHW_META:", HW_META)

## 2. Model load

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
MODEL_DTYPE = torch.bfloat16 if bf16_supported else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=MODEL_DTYPE,
    device_map="auto",
)
print("Model loaded:", MODEL_ID)
print(f"Params: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

## 4. Dataset — AIME

In [ ]:
from peft import LoraConfig, get_peft_model

# Registered adapter targets (preregistration v1.3)
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# autocast_adapter_dtype=False: skip dtype cast that fails on P100 with
# torch 2.10+cu128 (no kernel image for SM60).
model = get_peft_model(model, lora_config, autocast_adapter_dtype=False)
model.print_trainable_parameters()

## 4. Dataset - AMC

In [ ]:
from datasets import load_dataset

dataset = load_dataset("kaggle-aimo/amc_filtered")
train_data = dataset["train"]
print(f"Train size: {len(train_data)}")
print(train_data[0])

In [ ]:
import re

SYSTEM_PROMPT = (
    "You are a reasoning assistant. "
    "Solve the problem carefully and end with a final answer."
)

def format_example(example):
    prompt = str(example["task"])
    answer = str(example["answer"])
    prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": prompt},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    return {"prompt": prompt, "answer": answer}

train_data = train_data.map(format_example)
print(train_data[0]["prompt"][:300])

## 5. Reward function

In [ ]:
def extract_final_number(text):
    text_str = str(text)
    matches = re.findall(r"[-+]?\d+(?:\.\d+)?", text_str.replace(",", ""))
    return matches[-1] if matches else None


def final_number_exact_match(completions, answer, **kwargs):
    n_prompts = len(answer)
    assert len(completions) % n_prompts == 0, (
        f"len(completions)={len(completions)} not divisible by n_prompts={n_prompts}"
    )
    n_gen = len(completions) // n_prompts
    rewards = []
    for i, completion in enumerate(completions):
        pred = extract_final_number(completion)
        gold = extract_final_number(answer[i // n_gen])
        rewards.append(1.0 if pred is not None and pred == gold else 0.0)
    return rewards

## 6. GRPO training (pilot — short run)

In [ ]:
from transformers import TrainerCallback
import time as _time
from pathlib import Path

def _extract_logged_reward(logs):
    reward_keys = [
        "reward",
        "rewards/correctness_reward/mean",
        "rewards/final_number_exact_match/mean",
        "rewards/mcq_label_exact_match/mean",
        "rewards/mbpp_test_pass/mean",
        "rewards/humaneval_test_pass/mean",
    ]
    for key in reward_keys:
        value = logs.get(key)
        if value is not None:
            return value
    return float("nan")

class LiveProgressCallback(TrainerCallback):
    MIN_STEPS = 180
    ZERO_REWARD_WINDOW_STEPS = 80
    ZERO_REWARD_THRESHOLD = 5e-5
    NO_IMPROVE_PATIENCE_STEPS = 180
    IMPROVEMENT_DELTA = 2e-4
    MAX_RUNTIME_MINUTES = 480

    def __init__(self, max_steps):
        self._max = max_steps
        self._t0 = _time.time()
        self._reward_points = []
        self._best_reward = float("-inf")
        self._best_step = 0
        self._log_path = None

    def _ensure_log_path(self, args):
        if self._log_path is None:
            out_dir = getattr(args, "output_dir", None) or "."
            self._log_path = Path(out_dir) / "train_progress.log"
            self._log_path.parent.mkdir(parents=True, exist_ok=True)

    def _emit(self, args, message):
        print(message, flush=True)
        self._ensure_log_path(args)
        with self._log_path.open("a", encoding="utf-8") as fh:
            fh.write(message + "\n")
            fh.flush()

    def _request_stop(self, args, control, reason):
        if getattr(control, "should_training_stop", False):
            return
        control.should_training_stop = True
        self._emit(args, f"[EARLY-STOP] {reason}")

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        step = state.global_step
        reward = _extract_logged_reward(logs)
        loss = logs.get("loss", float("nan"))
        lr = logs.get("learning_rate", float("nan"))
        elapsed = _time.time() - self._t0
        eta = (elapsed / step * (self._max - step)) if step > 0 else float("nan")
        message = (
            f"[{step:4d}/{self._max}] reward={reward:.4f}  loss={loss:.5f}"
            f"  lr={lr:.2e}  elapsed={elapsed/60:.1f}m  eta={eta/60:.1f}m"
        )
        self._emit(args, message)

        if isinstance(loss, (int, float)) and (loss != loss or loss in (float("inf"), float("-inf"))):
            self._request_stop(args, control, f"non-finite loss at step={step}: loss={loss}")
            return

        if elapsed / 60.0 >= self.MAX_RUNTIME_MINUTES:
            self._request_stop(args, control, f"runtime {elapsed/60.0:.1f}m exceeded limit {self.MAX_RUNTIME_MINUTES}m at step={step}")
            return

        if not isinstance(reward, (int, float)) or reward != reward:
            return

        self._reward_points.append((step, float(reward)))
        cutoff_step = step - self.ZERO_REWARD_WINDOW_STEPS
        self._reward_points = [(s, r) for (s, r) in self._reward_points if s > cutoff_step]

        if reward > self._best_reward + self.IMPROVEMENT_DELTA:
            self._best_reward = float(reward)
            self._best_step = step

        if step < self.MIN_STEPS:
            return

        if self._reward_points:
            window_mean = sum(r for _, r in self._reward_points) / len(self._reward_points)
            if window_mean <= self.ZERO_REWARD_THRESHOLD:
                self._request_stop(args, control, f"reward mean {window_mean:.6f} <= {self.ZERO_REWARD_THRESHOLD} for last {self.ZERO_REWARD_WINDOW_STEPS} steps (step={step})")
                return

        if step - self._best_step >= self.NO_IMPROVE_PATIENCE_STEPS:
            self._request_stop(args, control, f"no reward improvement for {self.NO_IMPROVE_PATIENCE_STEPS} steps (best={self._best_reward:.6f} at step={self._best_step})")
            return

import time
from trl import GRPOConfig, GRPOTrainer

# Deviation from v1.3: max_completion_length=96 (registered default: 256)
# 64-token run (v5) yielded near-zero reward; 96 allows minimal reasoning chain
grpo_config = GRPOConfig(
    output_dir="/kaggle/working/amc-qwen2.5-1.5b-phase1",
    max_steps=1000,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_completion_length=96,
    learning_rate=1e-4,
    bf16=bf16_supported,
    fp16=not bf16_supported,
    logging_steps=10,
    save_steps=50,
    save_total_limit=1,
    report_to="none",
)

trainer = GRPOTrainer(
    callbacks=[LiveProgressCallback(grpo_config.max_steps)],
    model=model,
    args=grpo_config,
    train_dataset=train_data,
    reward_funcs=final_number_exact_match,
    processing_class=tokenizer,
)

t0 = time.time()
trainer.train()
total_time = time.time() - t0
actual_steps = max(trainer.state.global_step, 1)
step_time_sec = total_time / actual_steps

print(f"Training complete.")
print(f"Completed steps: {actual_steps}/{grpo_config.max_steps}")
print(f"Total time    : {total_time:.1f}s")
print(f"Per-step time : {step_time_sec:.2f}s  (on {HW_META['gpu_model']})")

## 7. Save adapter checkpoint

In [ ]:
ADAPTER_PATH = "/kaggle/working/amc-qwen2.5-1.5b-phase1/adapter"

model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"Adapter saved: {ADAPTER_PATH}")

from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=MODEL_DTYPE, device_map="auto")
loaded = PeftModel.from_pretrained(base, ADAPTER_PATH, autocast_adapter_dtype=False)
print("Adapter reload: OK")

## 8. Update vector extraction

In [ ]:
import numpy as np
import hashlib, json

REGISTERED_TARGETS = set(LORA_TARGET_MODULES)

def extract_registered_update_vector(peft_model):
    deltas = []
    layer_meta = []
    seen_targets = set()

    for name, module in peft_model.named_modules():
        if not (hasattr(module, "lora_A") and hasattr(module, "lora_B")):
            continue

        A = module.lora_A["default"].weight.detach().float().cpu().numpy()
        B = module.lora_B["default"].weight.detach().float().cpu().numpy()

        deltas.append(A.flatten())
        deltas.append(B.flatten())
        layer_meta.append({
            "name": name,
            "a_shape": list(A.shape),
            "b_shape": list(B.shape),
            "a_numel": int(A.size),
            "b_numel": int(B.size),
            "a_norm": float(np.linalg.norm(A)),
            "b_norm": float(np.linalg.norm(B)),
        })

        for target in REGISTERED_TARGETS:
            if name.endswith(target):
                seen_targets.add(target)

    if not deltas:
        raise ValueError("No LoRA layers found — check adapter config")

    missing = REGISTERED_TARGETS - seen_targets
    if missing:
        raise ValueError(f"Missing registered target modules: {missing}")

    vector = np.concatenate(deltas)
    return vector, layer_meta


update_vector, layer_meta = extract_registered_update_vector(loaded)

vector_bytes = update_vector.tobytes()
checksum = hashlib.sha256(vector_bytes).hexdigest()

print(f"Registered update vector shape : {update_vector.shape}")
print(f"Norm                          : {np.linalg.norm(update_vector):.4f}")
print(f"Non-zero ratio                : {(update_vector != 0).mean():.4f}")
print(f"SHA-256                       : {checksum}")
print(f"Layers captured               : {len(layer_meta)}")

VECTOR_PATH = "/kaggle/working/amc-qwen2.5-1.5b-phase1/update_vector.npy"
np.save(VECTOR_PATH, update_vector)

PROVENANCE_PATH = VECTOR_PATH.replace(".npy", "_provenance.json")
provenance = {
    "vector_path": VECTOR_PATH,
    "object_type": "registered_concat_lora_A_B",
    "sha256": checksum,
    "shape": list(update_vector.shape),
    "norm": float(np.linalg.norm(update_vector)),
    "registered_targets": sorted(REGISTERED_TARGETS),
    "layers": layer_meta,
}
with open(PROVENANCE_PATH, "w") as f:
    json.dump(provenance, f, indent=2)

print(f"\nRegistered update vector saved : {VECTOR_PATH}")
print(f"Provenance saved               : {PROVENANCE_PATH}")

## 9. SVD analysis — pilot diagnostic only

In [ ]:
def compute_r90(singular_values):
    """Number of singular values capturing 90% of total variance."""
    total = (singular_values ** 2).sum()
    cumvar = np.cumsum(singular_values ** 2) / total
    return int(np.searchsorted(cumvar, 0.90)) + 1

# Pilot-only diagnostic on dense effective layer deltas.
# This is separate from the registered multi-task H1a/H1b analysis,
# which operates on normalized concat(Delta W_A, Delta W_B) task vectors.
svd_deltas = []
for name, module in loaded.named_modules():
    if hasattr(module, "lora_A") and hasattr(module, "lora_B"):
        A = module.lora_A["default"].weight
        B = module.lora_B["default"].weight
        scaling = module.scaling["default"]
        delta = (scaling * (B @ A)).detach().float().cpu().numpy()
        svd_deltas.append((name, delta))

results = []
for name, delta in svd_deltas:
    U, S, Vh = np.linalg.svd(delta, full_matrices=False)
    r90 = compute_r90(S)
    results.append({"layer": name, "rank": delta.shape[1], "r90": r90, "s_max": float(S[0])})
    print(f"{name}: r90={r90}, s_max={S[0]:.4f}")

print("\nPilot SVD diagnostic: OK")

## 10. Run report

In [ ]:
import json, datetime

report = {
    "date": datetime.datetime.utcnow().isoformat(),
    "phase": 1,
    "model": MODEL_ID,
    "task": "AMC",
    "method": "GRPO",
    "hardware": HW_META,
    "training": {
        "max_steps": grpo_config.max_steps,
        "actual_steps": int(actual_steps),
        "total_time_sec": round(total_time, 1),
        "per_step_time_sec": round(step_time_sec, 2),
        "precision": "bf16" if bf16_supported else "fp16",
        "deviations": {
            "max_completion_length": {
                "registered": 256,
                "used": 96,
                "rationale": "64-token run (v5) yielded near-zero reward due to reasoning truncation; 96 allows minimal chain-of-thought",
            }
        },
    },
    "update_vector": {
        "object_type": "registered_concat_lora_A_B",
        "shape": list(update_vector.shape),
        "norm": float(np.linalg.norm(update_vector)),
        "sha256": checksum,
        "provenance_path": PROVENANCE_PATH,
        "layers_captured": len(layer_meta),
        "registered_targets": sorted(REGISTERED_TARGETS),
    },
    "svd_results": results,
    "analysis_scope": {
        "registered_primary_analysis_run": False,
        "pilot_svd_diagnostic_run": True,
    },
    "acceptance_criteria": {
        "model_loaded": True,
        "training_completed": True,
        "adapter_saved_and_reloaded": True,
        "update_vector_extracted": True,
        "update_vector_non_degenerate": bool(np.linalg.norm(update_vector) > 0),
        "registered_targets_all_covered": len(REGISTERED_TARGETS - {m["name"].split(".")[-1] for m in layer_meta}) == 0,
        "svd_diagnostic_ran": True,
    },
    "notes": "Phase 1 full run. max_completion_length=96 deviation from v1.3 (registered: 256). 64-token v5 run failed with near-zero reward. Label as exploratory in downstream analysis.",
}

REPORT_PATH = "/kaggle/working/amc-qwen2.5-1.5b-phase1/run_report.json"
with open(REPORT_PATH, "w") as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))
print("\nPhase 1 AMC run complete.")